# Proyecto Bank Marketing - Rol B: Transformación y Preparación de Datos

**Responsable**: Mauricio Tapia (Integrante 2)  
**Fecha**: Junio 2026  
**Asignatura**: ADY1100 – Preprocesamiento de Datos

---

## 📋 Resumen del Rol B

Este cuaderno documenta las **tareas de transformación y preparación de datos** que corresponden al Rol B:

- Escalamiento de variables numéricas con `StandardScaler`.
- Codificación de variables categóricas con `OneHotEncoder` y `OrdinalEncoder`.
- Construcción del pipeline con `ColumnTransformer` y `Pipeline`.
- Comparación antes/después de las transformaciones.
- Generación de 5 gráficos asociados a las transformaciones.

**Criterio fundamental**: Todas las transformaciones aprenden parámetros SOLO de `X_train` y se aplican a `X_train` y `X_test`.

---

## 📂 Estructura del Proyecto

```
bank_marketing/
├── data/                    # Datos originales y transformados
│   ├── bank_marketing.csv  # Dataset original
│   ├── X_train_original.csv
│   ├── X_test_original.csv
│   ├── X_train_transformed.csv
│   └── X_test_transformed.csv
├── docs/
│   ├── md-fuente/          # Documentos de referencia (md)
├── notebooks/
│   ├── role_a_cleanup.ipynb  # Rol A: Limpieza y separación train/test
│   ├── role_b_structure.ipynb # Rol B: Este cuaderno
│   └── role_c_features.ipynb  # Rol C: Feature engineering
├── reports/
│   ├── informe_final.ipynb  # Informe técnico consolidado
│   └── presentacion.pptx    # Presentación oral
└── README.md
```

## 1️⃣ Configurar Entorno Virtual y Dependencias

### 1.1 Instalar dependencias requeridas

En la terminal (fuera de este cuaderno), ejecutar:

```bash
# Navegar al directorio del proyecto
cd /home/mauricio/Documentos/GitHub/EDA-Bank-Marketing-Dataset./bank_marketing

# Crear entorno virtual
python3 -m venv venv
source venv/bin/activate

# Instalar paquetes
pip install pandas numpy scikit-learn matplotlib seaborn jupyter

# Generar requirements.txt
pip freeze > requirements.txt
```

### 1.2 Verificar que scikit-learn está disponible

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
import warnings
warnings.filterwarnings('ignore')

# Configurar estilos de gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Todas las dependencias importadas correctamente.")


---

## 2️⃣ Cargar Datos y Verificar Train/Test Split

### 2.1 Cargar el dataset original

Asegúrate de que el archivo `bank_marketing.csv` esté en `data/` antes de ejecutar la siguiente celda.  
Si no lo tienes, descargalo desde [Kaggle: Bank Marketing Dataset](https://www.kaggle.com/datasets/janiobachmann/bank-marketing-dataset).

In [ ]:
data_path = '../data/bank_marketing.csv'

try:
    df = pd.read_csv(data_path)
    print(f"✅ Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas.")
    print(f"\nPrimeras filas:")
    print(df.head(3))
except FileNotFoundError:
    print(f"❌ No se encontró {data_path}")
    print("Asegúrate de colocar el archivo en la carpeta 'data/'")


### 2.2 Definir columnas numéricas y categóricas

Basándonos en el acta del proyecto, separamos las variables en tres grupos:  
- **Numéricas**: Se escalarán con `StandardScaler`.
- **Categóricas nominales**: Se codificarán con `OneHotEncoder`.
- **Categóricas ordinales**: Se codificarán con `OrdinalEncoder`.

In [ ]:
numeric_features = [
    'age', 'campaign', 'pdays', 'previous',
    'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
    'euribor3m', 'nr.employed'
]

nominal_categorical_features = [
    'job', 'marital', 'contact', 'month', 'poutcome'
]

ordinal_categorical_features = [
    'education'
]

target_variable = 'y'

print(f"✅ Variables numéricas ({len(numeric_features)}): {numeric_features}")
print(f"✅ Variables categóricas nominales ({len(nominal_categorical_features)}): {nominal_categorical_features}")
print(f"✅ Variables categóricas ordinales ({len(ordinal_categorical_features)}): {ordinal_categorical_features}")


### 2.3 Realizar Train/Test Split con estratificación

**Nota importante**: Este split debe haberse realizado en el Rol A. Si aún no existe, lo realizamos aquí con estratificación según la variable objetivo.

In [ ]:
X = df.drop(columns=[target_variable])
y = df[target_variable]

RANDOM_STATE = 42
TEST_SIZE = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=TEST_SIZE, 
    stratify=y, 
    random_state=RANDOM_STATE
)

print(f"✅ Train/Test Split completado con random_state={RANDOM_STATE}")
print(f"   - X_train: {X_train.shape}")
print(f"   - X_test: {X_test.shape}")
print(f"   - y_train: {y_train.shape}")
print(f"   - y_test: {y_test.shape}")
print(f"\n   Proporción de clases en Train: {y_train.value_counts(normalize=True).to_dict()}")
print(f"   Proporción de clases en Test:  {y_test.value_counts(normalize=True).to_dict()}")

X_train.to_csv('../data/X_train_original.csv', index=False)
X_test.to_csv('../data/X_test_original.csv', index=False)
print(f"\n✅ Datasets originales guardados en data/")


---

## 3️⃣ Escalamiento de Variables Numéricas

### 3.1 Conceptos básicos

El escalamiento estandariza el rango de las variables numéricas, mejorando el desempeño de algoritmos de ML.

**Técnica**: `StandardScaler` 
- Centra en media = 0
- Escala con desviación estándar = 1
- Fórmula: $(x - \mu) / \sigma$

**Criterio fundamental**: El escalador debe hacer `fit()` **solo en `X_train`** y luego `transform()` en ambos conjuntos.

In [ ]:
scaler = StandardScaler()

X_train_numeric = X_train[numeric_features]
X_test_numeric = X_test[numeric_features]

print("📊 Antes del escalamiento (X_train):")
print(f"   Media: {X_train_numeric.mean().round(2).to_dict()}")
print(f"   Desv. Est: {X_train_numeric.std().round(2).to_dict()}")
print()

scaler.fit(X_train_numeric)

X_train_numeric_scaled = scaler.transform(X_train_numeric)
X_test_numeric_scaled = scaler.transform(X_test_numeric)

X_train_numeric_scaled_df = pd.DataFrame(X_train_numeric_scaled, columns=numeric_features)
X_test_numeric_scaled_df = pd.DataFrame(X_test_numeric_scaled, columns=numeric_features)

print("📊 Después del escalamiento (X_train escalado):")
print(f"   Media: {X_train_numeric_scaled_df.mean().round(4).to_dict()}")
print(f"   Desv. Est: {X_train_numeric_scaled_df.std().round(4).to_dict()}")
print()
print("✅ Escalamiento completado. Fit solo en Train, transform en Train y Test.")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gráfico 1: Distribución de una variable antes/después
var_plot = 'age'
axes[0, 0].hist(X_train_numeric[var_plot], bins=30, alpha=0.6, label='Original', color='blue')
axes[0, 0].set_title(f'Distribución de {var_plot} - ANTES del escalamiento', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel(var_plot)
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].legend()

axes[0, 1].hist(X_train_numeric_scaled_df[var_plot], bins=30, alpha=0.6, label='Escalado', color='green')
axes[0, 1].set_title(f'Distribución de {var_plot} - DESPUÉS del escalamiento', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel(f'{var_plot} (escalado)')
axes[0, 1].set_ylabel('Frecuencia')
axes[0, 1].legend()

# Gráfico 2: Boxplots antes/después
axes[1, 0].boxplot([X_train_numeric[var_plot]], labels=['Original'])
axes[1, 0].set_title(f'Boxplot {var_plot} - ANTES', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel(var_plot)

axes[1, 1].boxplot([X_train_numeric_scaled_df[var_plot]], labels=['Escalado'])
axes[1, 1].set_title(f'Boxplot {var_plot} - DESPUÉS', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel(f'{var_plot} (escalado)')

plt.suptitle('Gráfico 1: Comparación antes y después del escalamiento con StandardScaler', 
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../reports/01_escalamiento_comparacion.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico 1 guardado en reports/01_escalamiento_comparacion.png")


---

## 4️⃣ Codificación de Variables Categóricas

### 4.1 Conceptos básicos

**OneHotEncoding** convierte categorías nominales en columnas binarias.  
**OrdinalEncoding** mantiene el orden de categorías ordinales.

**Criterio fundamental**: Los codificadores deben hacer `fit()` **solo en `X_train`** para aprender las categorías existentes.

In [ ]:
one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False, drop='first')

X_train_nominal = X_train[nominal_categorical_features]
X_test_nominal = X_test[nominal_categorical_features]

print("📊 Variables categóricas nominales - ANTES de la codificación (X_train):")
print(X_train_nominal.head(3))
print()

one_hot_encoder.fit(X_train_nominal)

X_train_nominal_encoded = one_hot_encoder.transform(X_train_nominal)
X_test_nominal_encoded = one_hot_encoder.transform(X_test_nominal)

feature_names_nominal = one_hot_encoder.get_feature_names_out(nominal_categorical_features)
X_train_nominal_encoded_df = pd.DataFrame(X_train_nominal_encoded, columns=feature_names_nominal)
X_test_nominal_encoded_df = pd.DataFrame(X_test_nominal_encoded, columns=feature_names_nominal)

print("📊 Variables categóricas nominales - DESPUÉS de la codificación (X_train):")
print(X_train_nominal_encoded_df.head(3))
print(f"   Nuevas columnas: {len(feature_names_nominal)}")
print(f"   Ejemplos: {feature_names_nominal[:5]}")
print()
print("✅ OneHotEncoding completado. Fit solo en Train, transform en Train y Test.")


In [ ]:
ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_train_ordinal = X_train[ordinal_categorical_features]
X_test_ordinal = X_test[ordinal_categorical_features]

print("📊 Variables categóricas ordinales - ANTES de la codificación (X_train):")
print(X_train_ordinal['education'].value_counts())
print()

ordinal_encoder.fit(X_train_ordinal)

X_train_ordinal_encoded = ordinal_encoder.transform(X_train_ordinal)
X_test_ordinal_encoded = ordinal_encoder.transform(X_test_ordinal)

X_train_ordinal_encoded_df = pd.DataFrame(X_train_ordinal_encoded, columns=ordinal_categorical_features)
X_test_ordinal_encoded_df = pd.DataFrame(X_test_ordinal_encoded, columns=ordinal_categorical_features)

print("📊 Variables categóricas ordinales - DESPUÉS de la codificación (X_train):")
print(X_train_ordinal_encoded_df.head(10))
print()
print("✅ OrdinalEncoding completado. Fit solo en Train, transform en Train y Test.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

job_counts_before = X_train['job'].value_counts()
axes[0].barh(job_counts_before.index, job_counts_before.values, color='steelblue')
axes[0].set_title('Distribución de "job" - ANTES de codificación', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Frecuencia')

education_counts_before = X_train['education'].value_counts()
axes[1].barh(education_counts_before.index, education_counts_before.values, color='coral')
axes[1].set_title('Distribución de "education" (ordinal) - ANTES de codificación', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Frecuencia')

plt.suptitle('Gráfico 2: Distribución de variables categóricas nominales y ordinales', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/02_categoricas_antes.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico 2 guardado en reports/02_categoricas_antes.png")


---

## 5️⃣ Construcción del Pipeline

### 5.1 Pipeline automatizado con ColumnTransformer

El `ColumnTransformer` aplica transformaciones específicas a diferentes grupos de columnas de manera coordinada.

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

nominal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse=False, drop='first')),
    ]
)

ordinal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('nom', nominal_transformer, nominal_categorical_features),
        ('ord', ordinal_transformer, ordinal_categorical_features),
    ],
    remainder='drop',
)

print("✅ Pipeline creado con ColumnTransformer:")
print("   - Numeric transformer: SimpleImputer + StandardScaler")
print("   - Nominal transformer: SimpleImputer + OneHotEncoder")
print("   - Ordinal transformer: SimpleImputer + OrdinalEncoder")


In [ ]:
preprocessor.fit(X_train)

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"✅ Pipeline ejecutado:")
print(f"   - X_train shape: {X_train.shape} → {X_train_transformed.shape}")
print(f"   - X_test shape: {X_test.shape} → {X_test_transformed.shape}")
print()
print(f"✅ Verificación crítica: fit() se realizó SOLO en X_train")
print(f"   - Imputadores aprendieron parámetros de Train")
print(f"   - Escalador aprendió media y desviación estándar de Train")
print(f"   - Codificadores aprendieron categorías de Train")
print()
print(f"   Transform se aplicó a Train Y Test:")
print(f"   - X_train_transformed: {X_train_transformed.shape}")
print(f"   - X_test_transformed: {X_test_transformed.shape}")

X_train_transformed_df = pd.DataFrame(X_train_transformed)
X_test_transformed_df = pd.DataFrame(X_test_transformed)

X_train_transformed_df.to_csv('../data/X_train_transformed.csv', index=False)
X_test_transformed_df.to_csv('../data/X_test_transformed.csv', index=False)
print()
print("✅ Datasets transformados guardados en data/")


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

datasets = ['X_train\n(Original)', 'X_train\n(Transformado)', 'X_test\n(Original)', 'X_test\n(Transformado)']
shapes = [X_train.shape[1], X_train_transformed.shape[1], X_test.shape[1], X_test_transformed.shape[1]]
colors = ['#1f77b4', '#ff7f0e', '#1f77b4', '#ff7f0e']

bars = ax.bar(datasets, shapes, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Número de columnas', fontsize=12, fontweight='bold')
ax.set_title('Gráfico 3: Dimensionalidad antes y después de las transformaciones', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(shapes) * 1.2)

for bar, shape in zip(bars, shapes):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(shape)}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/03_dimensionalidad_pipeline.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico 3 guardado en reports/03_dimensionalidad_pipeline.png")


---

## 6️⃣ Validación del Pipeline

### 6.1 Verificar ausencia de valores nulos

In [ ]:
print("🔍 VALIDACIÓN DEL PIPELINE")
print("=" * 60)
print()

print("1. Verificar ausencia de valores nulos en X_train transformado:")
null_count_train = np.isnan(X_train_transformed).sum()
print(f"   Valores NaN en X_train: {null_count_train}")
print(f"   ✅ Validación pasada" if null_count_train == 0 else f"   ❌ Validación fallida")
print()

print("2. Verificar ausencia de valores nulos en X_test transformado:")
null_count_test = np.isnan(X_test_transformed).sum()
print(f"   Valores NaN en X_test: {null_count_test}")
print(f"   ✅ Validación pasada" if null_count_test == 0 else f"   ❌ Validación fallida")
print()

print("3. Verificar que X_train escalado tenga media ≈ 0 y std ≈ 1:")
train_means = X_train_transformed_df.iloc[:, :len(numeric_features)].mean()
train_stds = X_train_transformed_df.iloc[:, :len(numeric_features)].std()
print(f"   Media (primeras 3 numéricas): {train_means.head(3).values}")
print(f"   Desv. Est (primeras 3 numéricas): {train_stds.head(3).values}")
print(f"   ✅ Escalamiento correcto" if (abs(train_means.head(3)) < 0.1).all() and (abs(train_stds.head(3) - 1) < 0.1).all() else f"   ⚠️ Revisar escalamiento")
print()

print("4. Verificar consistencia dimensional entre Train y Test:")
print(f"   X_train transformado: {X_train_transformed.shape}")
print(f"   X_test transformado: {X_test_transformed.shape}")
print(f"   ✅ Consistencia pasada" if X_train_transformed.shape[1] == X_test_transformed.shape[1] else f"   ❌ Inconsistencia detectada")
print()

print("=" * 60)
print("✅ Todas las validaciones completadas.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

numeric_train_subset = X_train[numeric_features].iloc[:, :5]
corr_before = numeric_train_subset.corr()
sns.heatmap(corr_before, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[0], cbar_kws={'label': 'Correlación'})
axes[0].set_title('Gráfico 4a: Matriz de correlación - ANTES del escalamiento', fontsize=12, fontweight='bold')

numeric_transformed_subset = X_train_transformed_df.iloc[:, :5]
corr_after = numeric_transformed_subset.corr()
sns.heatmap(corr_after, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1], cbar_kws={'label': 'Correlación'})
axes[1].set_title('Gráfico 4b: Matriz de correlación - DESPUÉS del escalamiento', fontsize=12, fontweight='bold')

plt.suptitle('Gráfico 4: Correlación entre variables antes y después de transformación', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/04_correlacion_transformacion.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico 4 guardado en reports/04_correlacion_transformacion.png")


In [ ]:
variance_before = X_train[numeric_features].var()
variance_after = X_train_transformed_df.iloc[:, :len(numeric_features)].var()

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

x = np.arange(len(numeric_features))
width = 0.35

bars1 = ax.bar(x - width/2, variance_before.values, width, label='Antes del escalamiento', alpha=0.7, color='#d62728')
bars2 = ax.bar(x + width/2, variance_after.values, width, label='Después del escalamiento', alpha=0.7, color='#2ca02c')

ax.set_xlabel('Variables numéricas', fontsize=12, fontweight='bold')
ax.set_ylabel('Varianza', fontsize=12, fontweight='bold')
ax.set_title('Gráfico 5: Varianza antes y después del escalamiento', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(numeric_features, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/05_varianza_escalamiento.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico 5 guardado en reports/05_varianza_escalamiento.png")


---

## 7️⃣ Guión para la Defensa Oral (Rol B - 33% del tiempo)

### 📝 Puntos principales a cubrir

#### 1. Escalamiento de variables numéricas (5-7 minutos)

**¿Qué hiciste?**
- Aplicaste `StandardScaler` a 9 variables numéricas.
- El escalador **SOLO** aprendió media y desviación estándar de `X_train`.
- Luego aplicaste la transformación a `X_train` y `X_test`.

**¿Por qué?**
- StandardScaler centra los datos en media 0 y los ajusta a desviación estándar 1.
- Esto es crítico para algoritmos de ML sensibles a la magnitud de features (por ejemplo: K-NN, SVM, regresión logística).
- Evitar ajustar el escalador en Test previene **data leakage**.

**¿Qué cambió?**
- Muestra Gráfico 1 (antes/después de `age` o similar).
- Muestra Gráfico 5 (varianza antes/después de todas las numéricas).
- Explica que la varianza homogénea mejora el desempeño del modelo.

---

#### 2. Codificación de variables categóricas (5-7 minutos)

**¿Qué hiciste?**
- Aplicaste `OneHotEncoder` a 5 variables nominales (`job`, `marital`, `contact`, `month`, `poutcome`).
- Aplicaste `OrdinalEncoder` a 1 variable ordinal (`education`).
- Ambos codificadores **SOLO** aprendieron las categorías en `X_train`.

**¿Por qué**?
- **OneHotEncoding**: Convierte categorías en columnas binarias. Ideal para variables sin orden natural.
- **OrdinalEncoding**: Asigna números a categorías ordenadas. Ideal para variables con jerarquía (ej: educación).
- `handle_unknown='ignore'` en OneHotEncoder previene errores si aparece una categoría nueva en Test.

**¿Qué cambió?**
- Muestra Gráfico 2 (distribución de categorías antes y después).
- Explica cómo se transformaron ejemplos concretos (ej: `job='admin.'` → columnas binarias).

---

#### 3. Pipeline y prevención de leakage (5-7 minutos)

**¿Qué hiciste?**
- Construiste un `ColumnTransformer` que coordina:
  - Imputación + Escalamiento de numéricas.
  - Imputación + OneHotEncoding de nominales.
  - Imputación + OrdinalEncoding de ordinales.
- El pipeline hizo `fit()` solo en `X_train` y luego `transform()` en ambos.

**¿Por qué?**
- Automatiza y evita errores manuales.
- Garantiza reproducibilidad.
- **Crítico para reproducibilidad**: Cualquiera puede aplicar exactamente el mismo pipeline a nuevos datos.

**¿Qué cambió?**
- Muestra Gráfico 3 (dimensionalidad antes/después).
- Explica que pasamos de 21 columnas a N columnas tras la codificación one-hot.
- Demuestra que Train y Test tienen la misma dimensión final.

---

### 🎯 Preguntas esperadas de los docentes

1. **¿Por qué hacer fit solo en Train?**
   - Respuesta: Para evitar data leakage. Si el escalador aprende estadísticas del Test, el modelo vería información del futuro.

2. **¿Qué pasaría si escalamos el dataset completo?**
   - Respuesta: El Test tendría información del Train (leakage), métricas serían optimistas e inválidas.

3. **¿Por qué StandardScaler en lugar de MinMaxScaler?**
   - Respuesta: StandardScaler es más robusto a outliers y es el estándar en ML. MinMaxScaler es sensible a valores extremos.

4. **¿Qué hace `handle_unknown='ignore'`?**
   - Respuesta: Si en Test aparece una categoría nueva no vista en Train, la trata como valor faltante en lugar de errores.

---

## 📊 Resumen de gráficos entregados

- **Gráfico 1**: Comparación antes/después del escalamiento.
- **Gráfico 2**: Distribución de variables categóricas nominales/ordinales.
- **Gráfico 3**: Dimensionalidad antes y después (barras).
- **Gráfico 4**: Matriz de correlación antes/después.
- **Gráfico 5**: Varianza de numéricas antes/después.

---

## ✅ Checklist de entrega Rol B

- [x] Escalamiento con `StandardScaler` (fit en Train, transform en Train+Test).
- [x] Codificación OneHot para nominales.
- [x] Codificación Ordinal para ordinales.
- [x] Pipeline con `ColumnTransformer`.
- [x] Validación de ausencia de NaN.
- [x] Validación de dimensionalidad consistente.
- [x] 5 gráficos de comparación antes/después.
- [x] Documentación clara en este notebook.
- [x] Guión de defensa oral preparado.

---

**Próximos pasos:**
- Revisar con el Rol A (datos limpios y preparados).
- Coordinar con el Rol C (feature engineering basado en dataset transformado).
- Consolidar en un informe final (`reports/informe_final.ipynb`).